In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the project path
import os

PROJECT_DIR = "/content/drive/MyDrive/DLP2"

RAW_DIR = os.path.join(PROJECT_DIR, "data/raw")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data/processed")

print(PROJECT_DIR)

/content/drive/MyDrive/DLP2


In [ ]:
# Download dataset into Drive
import zipfile
import urllib.request

url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

zip_path = os.path.join(RAW_DIR, "uci_har.zip")

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
    print("Dataset downloaded.")
else:
    print("Dataset already exists.")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(RAW_DIR)

print("Dataset extracted.")

Dataset downloaded.
Dataset extracted.


In [ ]:
# Extract dataset
import zipfile
import os

RAW_DIR = "/content/drive/MyDrive/DLP2/data/raw"

zip_path = os.path.join(RAW_DIR, "UCI HAR Dataset.zip")

extract_path = RAW_DIR

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed.")
print(os.listdir(RAW_DIR))

Extraction completed.
['uci_har.zip', 'UCI HAR Dataset.names', 'UCI HAR Dataset.zip', 'UCI HAR Dataset', '__MACOSX']


In [ ]:
# Verify dataset exists
dataset_path = os.path.join(RAW_DIR, "UCI HAR Dataset")

print(os.listdir(dataset_path))

['.DS_Store', 'activity_labels.txt', 'features.txt', 'features_info.txt', 'README.txt', 'test', 'train']


In [ ]:
import numpy as np
import pandas as pd
import os

SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]

def load_signals(split):
    signal_data = []

    for signal in SIGNALS:
        file_path = os.path.join(
            dataset_path,
            split,
            "Inertial Signals",
            f"{signal}_{split}.txt"
        )

        data = pd.read_csv(file_path, sep=r"\s+", header=None)
        signal_data.append(data.values)

    return np.transpose(np.array(signal_data), (1, 2, 0))


def load_labels(split):
    file_path = os.path.join(dataset_path, split, f"y_{split}.txt")
    labels = pd.read_csv(file_path, sep=r"\s+", header=None).values.flatten()

    return labels - 1


X_train = load_signals("train")
y_train = load_labels("train")

X_test = load_signals("test")
y_test = load_labels("test")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (7352, 128, 9)
y_train: (7352,)
X_test: (2947, 128, 9)
y_test: (2947,)


In [ ]:
# Create validation split
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Train:", X_train_final.shape, y_train_final.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (5881, 128, 9) (5881,)
Val: (1471, 128, 9) (1471,)
Test: (2947, 128, 9) (2947,)


In [ ]:
# Normalize data
mean = X_train_final.mean(axis=(0, 1), keepdims=True)
std = X_train_final.std(axis=(0, 1), keepdims=True)

X_train_norm = (X_train_final - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

print("Train mean:", X_train_norm.mean())
print("Train std:", X_train_norm.std())

Train mean: 1.3067863446153763e-16
Train std: 1.0000000000000002


In [ ]:
# Save processed files
import os
import numpy as np
import json

PROCESSED_DIR = "/content/drive/MyDrive/DLP2/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

np.save(os.path.join(PROCESSED_DIR, "X_train.npy"), X_train_norm)
np.save(os.path.join(PROCESSED_DIR, "X_val.npy"), X_val_norm)
np.save(os.path.join(PROCESSED_DIR, "X_test.npy"), X_test_norm)

np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y_train_final)
np.save(os.path.join(PROCESSED_DIR, "y_val.npy"), y_val)
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"), y_test)

activity_labels = {
    0: "WALKING",
    1: "WALKING_UPSTAIRS",
    2: "WALKING_DOWNSTAIRS",
    3: "SITTING",
    4: "STANDING",
    5: "LAYING"
}

with open(os.path.join(PROCESSED_DIR, "activity_labels.json"), "w") as f:
    json.dump(activity_labels, f, indent=4)

print("All processed files saved.")
print(os.listdir(PROCESSED_DIR))

All processed files saved.
['X_train.npy', 'X_val.npy', 'X_test.npy', 'y_train.npy', 'y_val.npy', 'y_test.npy', 'activity_labels.json']
